# Credit Card Activity Video → Interactive Dashboard

Take a screen recording (`.mp4`) of a credit card activity screen (e.g. Wealthsimple, TD, etc.) and produce:

1. A clean list of unique transactions extracted via OCR
2. Summary statistics by category, merchant, and day
3. An interactive HTML dashboard with cumulative spend, daily activity, category share, top merchants, and a filterable transactions table

## How it works

1. **Frame extraction** — pull frames from the video at 2 fps
2. **Deduplication** — use perceptual hashing to keep only frames that show meaningfully different content (i.e. the user actually scrolled)
3. **OCR** — run Tesseract on each unique frame to extract text
4. **Parsing** — regex out date headers and `Merchant — $X.XX CAD` lines
5. **Vote-based dedup** — the same transaction appears in many frames as the user scrolls. For each `(merchant, amount)` pair, the date with the most frame-votes wins. This rejects misattributions that occur when a date header has scrolled just off the top of the screen.
6. **Rendering** — generate a self-contained HTML dashboard with Chart.js graphs and a filterable table

## Prerequisites

On the system:
```bash
sudo apt install tesseract-ocr ffmpeg   # Linux
brew install tesseract ffmpeg           # macOS
```

Python packages: `pip install pytesseract pillow imagehash`

## 1. Configuration

In [ ]:
# === EDIT THESE ===
VIDEO_PATH = '/mnt/user-data/uploads/Credit_card.MP4'   # path to your screen recording
OUTPUT_HTML = 'spending_dashboard.html'                  # where to write the dashboard
FRAMES_PER_SECOND = 2     # higher = more frames = more thorough, slower
PHASH_DISTANCE = 4        # 0-8: smaller = stricter dedup (more frames kept)
VOTE_THRESHOLD = 2        # min frames a transaction must appear in to count (lower = more permissive)

# Optional: override the recording date (otherwise pulled from video metadata)
# Format: 'YYYY-MM-DD'
RECORDING_DATE_OVERRIDE = None

## 2. Extract frames

In [ ]:
import subprocess, shutil, tempfile
from pathlib import Path

frames_dir = Path(tempfile.mkdtemp(prefix='cc_frames_'))
subprocess.run([
    'ffmpeg', '-i', VIDEO_PATH, '-vf', f'fps={FRAMES_PER_SECOND}',
    str(frames_dir / 'f_%04d.png'), '-y', '-loglevel', 'error'
], check=True)

frame_paths = sorted(frames_dir.glob('*.png'))
print(f'Extracted {len(frame_paths)} frames to {frames_dir}')

## 3. Deduplicate near-identical frames

Uses perceptual hashing — frames that look almost identical (because the user paused mid-scroll) get collapsed. This dramatically cuts OCR time without losing transactions.

In [ ]:
from PIL import Image
import imagehash

unique_frames = []
seen_hashes = []
for fp in frame_paths:
    h = imagehash.phash(Image.open(fp), hash_size=16)
    if all(h - sh > PHASH_DISTANCE for sh in seen_hashes):
        unique_frames.append(fp)
        seen_hashes.append(h)

print(f'Kept {len(unique_frames)} unique frames (dropped {len(frame_paths) - len(unique_frames)} near-duplicates)')

## 4. Determine the recording date

Needed to resolve "YESTERDAY" / "TODAY" date headers. Pulled from video metadata unless overridden.

In [ ]:
from datetime import datetime, timedelta

if RECORDING_DATE_OVERRIDE:
    recording_dt = datetime.strptime(RECORDING_DATE_OVERRIDE, '%Y-%m-%d')
else:
    out = subprocess.run([
        'ffprobe', '-v', 'error', '-show_entries',
        'format_tags=creation_time', '-of', 'default=nw=1:nk=1', VIDEO_PATH
    ], capture_output=True, text=True)
    try:
        recording_dt = datetime.strptime(out.stdout.strip()[:19], '%Y-%m-%dT%H:%M:%S')
    except ValueError:
        recording_dt = datetime.now()
        print('Could not parse video creation time, using now')

print(f'Recording date: {recording_dt.date()}')

## 5. OCR + parse each frame

Walk each frame top-to-bottom: a date header (e.g. `MAY 11, 2026`, `YESTERDAY`) becomes the current date; subsequent transaction lines (`Merchant — $X.XX CAD`) get tagged with that date.

In [ ]:
import re
import pytesseract

DATE_HEADER = re.compile(r'^(YESTERDAY|TODAY|[A-Z]{3,9}\s+\d{1,2},?\s+\d{4})\s*$', re.IGNORECASE)
TX_LINE = re.compile(r'^(.+?)\s+[-—–]+\s*\$?([\d,]+\.\d{2})\s*CAD\s*$')
PAYMENT_LINE = re.compile(r'^(.*Credit card payment.*?)\s+\$?([\d,]+\.\d{2})\s*CAD\s*$', re.IGNORECASE)

# OCR commonly misreads the iOS card-icon glyph as one of these leading tokens:
# =, @, ©, ®, $, S, s, Ss, 8, =), (=), (os), (@), (s), [*], Gp, Gs
ICON_JUNK = re.compile(r'^([=@©®$()\[\]\*]+|[Ss]{1,2}|[Gg][ps]|\d+)\s+')

def parse_date(label, rec_dt):
    label = label.strip().upper()
    if label == 'YESTERDAY':
        return (rec_dt - timedelta(days=1)).date().isoformat()
    if label == 'TODAY':
        return rec_dt.date().isoformat()
    for fmt in ('%B %d, %Y', '%b %d, %Y', '%B %d %Y', '%b %d %Y'):
        try:
            return datetime.strptime(label, fmt).date().isoformat()
        except ValueError:
            continue
    return None

def clean_merchant(m):
    m = m.strip()
    while True:
        new = ICON_JUNK.sub('', m)
        if new == m:
            break
        m = new
    return m.strip()

per_frame_items = []
for fp in unique_frames:
    text = pytesseract.image_to_string(Image.open(fp))
    current_date = None
    items = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line or line.lower() in {'purchase', 'pending', 'from: chequing'}:
            continue
        m_date = DATE_HEADER.match(line)
        if m_date:
            d = parse_date(m_date.group(1), recording_dt)
            if d:
                current_date = d
            continue
        if current_date is None:
            continue
        if PAYMENT_LINE.match(line):
            continue   # credit card payments are not spend
        m_tx = TX_LINE.match(line)
        if m_tx:
            merchant = clean_merchant(m_tx.group(1))
            if len(merchant) < 3:
                continue
            try:
                amount = float(m_tx.group(2).replace(',', ''))
            except ValueError:
                continue
            items.append((current_date, merchant, amount))
    per_frame_items.append(items)

total_raw = sum(len(items) for items in per_frame_items)
print(f'Parsed {total_raw} transaction lines across {len(unique_frames)} frames (with many duplicates from scroll overlap)')

## 6. Vote-based dedup

Each transaction appears in many frames as the user scrolls. For each `(merchant, amount)` pair, count how many frames place it under each date. The legitimate date appears in many frames; dates that result from a header scrolling just off-screen appear in only one or two frames.

Amazon order IDs are normalized for grouping (since each order has a unique random ID, OCR noise doesn't fragment them).

In [ ]:
from collections import Counter, defaultdict

def normalize_key(merchant):
    if re.match(r'^Amzn\s+Mktp', merchant, re.IGNORECASE):
        return 'AMAZON'
    return merchant.lower()

vote = Counter()       # (norm_key, amount, date) -> frame count
display_name = {}      # (norm_key, amount, date) -> best display string

for items in per_frame_items:
    seen_this_frame = set()
    for d, m, a in items:
        k = (normalize_key(m), a, d)
        if k in seen_this_frame:
            continue
        seen_this_frame.add(k)
        vote[k] += 1
        if len(m) > len(display_name.get(k, '')):
            display_name[k] = m

# Group by (norm_key, amount) and keep date(s) with enough support
grouped = defaultdict(list)
for (km, a, d), c in vote.items():
    grouped[(km, a)].append((d, c))

transactions = []
for (km, a), date_counts in grouped.items():
    max_c = max(c for _, c in date_counts)
    if max_c == 1:
        # very short video or item only seen once — keep all
        for d, c in date_counts:
            transactions.append((d, display_name[(km, a, d)], a))
    else:
        for d, c in date_counts:
            if c >= max(VOTE_THRESHOLD, max_c / 2):
                transactions.append((d, display_name[(km, a, d)], a))

transactions.sort(key=lambda t: (t[0], -t[2], t[1]))
print(f'Final unique transactions: {len(transactions)}')
print(f'Total spend: ${sum(t[2] for t in transactions):,.2f}')

## 7. Categorize transactions

Lightweight rule-based categorizer. Customize the keyword lists below for your own merchants.

In [ ]:
CATEGORY_RULES = [
    ('Amazon',          ['amzn', 'amazon']),
    ('Professional',    ['engineers nova scotia', 'engineers ns', 'p.eng']),
    ('Subscriptions',   ['surfshark', 'koodo', 'apple.com', 'icloud', 'spotify', 'netflix', 'youtube', 'github', 'openai', 'anthropic']),
    ('Restaurants',     ['poke', 'broth', 'ubereats', 'crepe', 'skip', 'doordash', 'tim hortons', 'starbucks', 'mcdonald']),
    ('Discount retail', ['dollarama', 'dollar tree', 'giant tiger']),
    ('Transportation',  ['communauto', 'uber canada', 'lyft', 'transit', 'parking', 'esso', 'shell', 'petro']),
    ('Home',            ['ikea', 'home depot', 'rona', 'canadian tire']),
    ('Groceries',       ['grocery', 'loblaws', 'sobeys', 'superstore', 'atlantic superstore', 'walmart', 'plaza', 'no frills']),
]

def categorize(merchant):
    m = merchant.lower()
    # Special-case: Uber Canada/Ubereats is restaurants, not transportation
    if 'uber canada' in m and 'ubereats' in m:
        return 'Restaurants'
    for cat, keywords in CATEGORY_RULES:
        if any(k in m for k in keywords):
            return cat
    return 'Other'

tx_with_cat = [(d, m, a, categorize(m)) for d, m, a in transactions]
for t in tx_with_cat:
    print(f'  {t[0]}  {t[1]:<40s}  ${t[2]:>8.2f}  [{t[3]}]')

## 8. Quick summaries

In [ ]:
import pandas as pd

df = pd.DataFrame(tx_with_cat, columns=['date', 'merchant', 'amount', 'category'])

print('=== TOTAL ===')
print(f'${df.amount.sum():,.2f} across {len(df)} transactions')
print(f'Date range: {df.date.min()} to {df.date.max()}')
print()
print('=== BY CATEGORY ===')
print(df.groupby('category').amount.agg(['sum', 'count']).sort_values('sum', ascending=False).round(2))
print()
print('=== TOP MERCHANTS (Amazon grouped) ===')
df['merchant_grp'] = df['merchant'].apply(
    lambda m: 'Amazon' if m.lower().startswith('amzn') else m
)
print(df.groupby('merchant_grp').amount.agg(['sum', 'count']).sort_values('sum', ascending=False).head(10).round(2))
print()
print('=== DAILY ===')
print(df.groupby('date').amount.agg(['sum', 'count']).round(2))

## 9. Generate the HTML dashboard

Self-contained HTML with Chart.js loaded from CDN. Just open the output file in any browser.

In [ ]:
import json

CAT_COLORS = {
    'Amazon': '#d4a574', 'Professional': '#c4554f', 'Subscriptions': '#6ea8c4',
    'Restaurants': '#d97757', 'Discount retail': '#7cb992', 'Transportation': '#9a7ab8',
    'Home': '#b89968', 'Groceries': '#5e9b8a', 'Other': '#4a5468',
}

# KPIs
total_spend = df.amount.sum()
n_tx = len(df)
n_days = (pd.to_datetime(df.date.max()) - pd.to_datetime(df.date.min())).days + 1
daily_avg = total_spend / n_days
largest = df.loc[df.amount.idxmax()]
median_charge = df.amount.median()
date_range = f'{df.date.min()[5:]} to {df.date.max()[5:]}'

tx_json = json.dumps([[r.date, r.merchant, r.amount, r.category] for _, r in df.iterrows()])
cat_colors_json = json.dumps(CAT_COLORS)

html = f'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Credit Card Spending Dashboard</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,400;9..144,500;9..144,600&family=JetBrains+Mono:wght@400;500;600&family=Inter:wght@400;500;600&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<style>
  :root {{ --bg:#0a0e14; --panel:#11161f; --panel-2:#161c27; --ink:#e8ecf2; --ink-dim:#8a94a6; --ink-faint:#4a5468; --rule:#1f2632; --accent:#d4a574; --grid:rgba(138,148,166,0.08); }}
  * {{ box-sizing:border-box; margin:0; padding:0; }}
  html,body {{ background:var(--bg); color:var(--ink); font-family:'Inter',sans-serif; line-height:1.5; -webkit-font-smoothing:antialiased; }}
  body {{ background:radial-gradient(ellipse 80% 50% at 10% 0%,rgba(212,165,116,0.06),transparent 60%),radial-gradient(ellipse 60% 40% at 90% 100%,rgba(110,168,196,0.05),transparent 60%),var(--bg); min-height:100vh; padding:48px 32px 80px; }}
  .container {{ max-width:1280px; margin:0 auto; }}
  header {{ margin-bottom:48px; border-bottom:1px solid var(--rule); padding-bottom:32px; }}
  .eyebrow {{ font-family:'JetBrains Mono',monospace; font-size:11px; letter-spacing:0.18em; text-transform:uppercase; color:var(--accent); margin-bottom:12px; }}
  h1 {{ font-family:'Fraunces',serif; font-weight:500; font-size:clamp(32px,4.5vw,56px); letter-spacing:-0.02em; line-height:1.05; margin-bottom:8px; }}
  h1 em {{ font-style:italic; color:var(--accent); font-weight:400; }}
  .subtitle {{ color:var(--ink-dim); font-size:15px; max-width:680px; }}
  .kpis {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(180px,1fr)); gap:1px; background:var(--rule); border:1px solid var(--rule); margin-bottom:32px; }}
  .kpi {{ background:var(--panel); padding:24px 28px; }}
  .kpi-label {{ font-family:'JetBrains Mono',monospace; font-size:10px; letter-spacing:0.15em; text-transform:uppercase; color:var(--ink-dim); margin-bottom:12px; }}
  .kpi-value {{ font-family:'Fraunces',serif; font-size:32px; font-weight:500; letter-spacing:-0.02em; line-height:1; }}
  .kpi-sub {{ font-family:'JetBrains Mono',monospace; font-size:11px; color:var(--ink-faint); margin-top:8px; }}
  .kpi-value.accent {{ color:var(--accent); }}
  section {{ margin-bottom:48px; }}
  .section-head {{ display:flex; align-items:baseline; justify-content:space-between; margin-bottom:20px; padding-bottom:12px; border-bottom:1px solid var(--rule); }}
  .section-head h2 {{ font-family:'Fraunces',serif; font-weight:500; font-size:24px; letter-spacing:-0.01em; }}
  .section-head .tag {{ font-family:'JetBrains Mono',monospace; font-size:10px; letter-spacing:0.15em; color:var(--ink-faint); text-transform:uppercase; }}
  .grid-3 {{ display:grid; grid-template-columns:2fr 1fr; gap:24px; }}
  @media (max-width:880px) {{ .grid-3 {{ grid-template-columns:1fr; }} }}
  .card {{ background:var(--panel); border:1px solid var(--rule); padding:28px; }}
  .card-title {{ font-family:'JetBrains Mono',monospace; font-size:11px; letter-spacing:0.15em; text-transform:uppercase; color:var(--ink-dim); margin-bottom:4px; }}
  .card-headline {{ font-family:'Fraunces',serif; font-size:20px; font-weight:500; margin-bottom:24px; letter-spacing:-0.01em; }}
  .chart-wrap {{ position:relative; height:320px; }}
  .chart-wrap.tall {{ height:380px; }}
  .cat-list {{ display:flex; flex-direction:column; gap:14px; }}
  .cat-row {{ display:grid; grid-template-columns:140px 1fr 80px; gap:12px; align-items:center; }}
  .cat-name {{ font-size:13px; color:var(--ink); font-weight:500; }}
  .cat-bar-track {{ height:6px; background:rgba(138,148,166,0.1); }}
  .cat-bar-fill {{ height:100%; background:var(--accent); transition:width 0.6s cubic-bezier(.2,.7,.3,1); }}
  .cat-amount {{ text-align:right; font-family:'JetBrains Mono',monospace; font-size:12px; color:var(--ink); }}
  .cat-pct {{ font-family:'JetBrains Mono',monospace; font-size:10px; color:var(--ink-faint); }}
  .table-wrap {{ background:var(--panel); border:1px solid var(--rule); overflow:hidden; }}
  .table-controls {{ display:flex; gap:6px; padding:16px 20px; border-bottom:1px solid var(--rule); flex-wrap:wrap; }}
  .chip {{ font-family:'JetBrains Mono',monospace; font-size:10px; letter-spacing:0.1em; text-transform:uppercase; padding:6px 12px; background:transparent; border:1px solid var(--rule); color:var(--ink-dim); cursor:pointer; transition:all 0.15s; }}
  .chip:hover {{ color:var(--ink); border-color:var(--ink-faint); }}
  .chip.active {{ background:var(--accent); color:var(--bg); border-color:var(--accent); font-weight:600; }}
  table {{ width:100%; border-collapse:collapse; }}
  thead {{ font-family:'JetBrains Mono',monospace; font-size:10px; letter-spacing:0.12em; text-transform:uppercase; color:var(--ink-faint); }}
  th {{ text-align:left; padding:14px 20px; border-bottom:1px solid var(--rule); font-weight:500; }}
  th.num {{ text-align:right; }}
  tbody tr {{ border-bottom:1px solid rgba(31,38,50,0.5); transition:background 0.15s; }}
  tbody tr:hover {{ background:var(--panel-2); }}
  tbody tr.hidden {{ display:none; }}
  td {{ padding:14px 20px; font-size:13px; }}
  td.num {{ text-align:right; font-family:'JetBrains Mono',monospace; }}
  td.date {{ font-family:'JetBrains Mono',monospace; color:var(--ink-dim); font-size:12px; }}
  .cat-tag {{ display:inline-block; font-family:'JetBrains Mono',monospace; font-size:9.5px; letter-spacing:0.1em; text-transform:uppercase; padding:3px 8px; border:1px solid var(--rule); color:var(--ink-dim); }}
  footer {{ margin-top:64px; padding-top:24px; border-top:1px solid var(--rule); font-family:'JetBrains Mono',monospace; font-size:11px; color:var(--ink-faint); display:flex; justify-content:space-between; flex-wrap:wrap; gap:12px; }}
</style>
</head>
<body>
<div class="container">
  <header>
    <div class="eyebrow">Credit Card Activity &middot; {date_range}</div>
    <h1>{n_days} days, <em>{n_tx} charges</em>.</h1>
    <p class="subtitle">Extracted from screen recording via OCR. Filter the table below by tapping a category.</p>
  </header>
  <div class="kpis">
    <div class="kpi"><div class="kpi-label">Total Spend</div><div class="kpi-value accent">${total_spend:,.2f}</div><div class="kpi-sub">CAD &middot; {n_tx} transactions</div></div>
    <div class="kpi"><div class="kpi-label">Daily Avg</div><div class="kpi-value">${daily_avg:,.2f}</div><div class="kpi-sub">across {n_days} days</div></div>
    <div class="kpi"><div class="kpi-label">Largest Charge</div><div class="kpi-value">${largest.amount:,.2f}</div><div class="kpi-sub">{largest.merchant[:30]} &middot; {largest.date[5:]}</div></div>
    <div class="kpi"><div class="kpi-label">Median Charge</div><div class="kpi-value">${median_charge:,.2f}</div><div class="kpi-sub">typical purchase size</div></div>
  </div>
  <section><div class="section-head"><h2>Cumulative spend</h2><span class="tag">Running total</span></div>
    <div class="card"><div class="card-title">How the balance built up</div><div class="card-headline">Where the curve steepens.</div><div class="chart-wrap tall"><canvas id="cumChart"></canvas></div></div></section>
  <section class="grid-3">
    <div class="card"><div class="card-title">Daily activity</div><div class="card-headline">Spend by day.</div><div class="chart-wrap"><canvas id="dailyChart"></canvas></div></div>
    <div class="card"><div class="card-title">Category share</div><div class="card-headline">Where the money went.</div><div class="chart-wrap"><canvas id="catDonut"></canvas></div></div>
  </section>
  <section style="margin-top:48px;"><div class="section-head"><h2>By category</h2><span class="tag">Sorted by spend</span></div><div class="card"><div class="cat-list" id="catList"></div></div></section>
  <section><div class="section-head"><h2>Top merchants</h2><span class="tag">Amazon orders grouped</span></div><div class="card"><div class="chart-wrap tall"><canvas id="merchChart"></canvas></div></div></section>
  <section><div class="section-head"><h2>All transactions</h2><span class="tag">Tap a category to filter</span></div>
    <div class="table-wrap"><div class="table-controls" id="filters"></div>
      <table><thead><tr><th style="width:90px;">Date</th><th>Merchant</th><th>Category</th><th class="num">Amount</th></tr></thead><tbody id="txBody"></tbody></table>
    </div></section>
  <footer><span>Source: credit card activity screen recording &middot; OCR-extracted</span><span>Generated &middot; {datetime.now().strftime('%Y-%m-%d')}</span></footer>
</div>
<script>
const TX = {tx_json};
const CAT_COLORS = {cat_colors_json};
const INK = '#e8ecf2', DIM = '#8a94a6', GRID = 'rgba(138,148,166,0.08)';
const FONT = "'JetBrains Mono', monospace";
Chart.defaults.color = DIM; Chart.defaults.font.family = 'Inter, sans-serif'; Chart.defaults.font.size = 11;

const sorted = [...TX].sort((a,b) => a[0].localeCompare(b[0]));
const dailyMap = {{}};
sorted.forEach(t => {{ dailyMap[t[0]] = (dailyMap[t[0]] || 0) + t[2]; }});
const days = Object.keys(dailyMap).sort();
let running = 0; const cum = days.map(d => {{ running += dailyMap[d]; return running; }});

const tooltipStyle = {{ backgroundColor: '#161c27', borderColor: '#1f2632', borderWidth: 1, titleFont: {{ family: FONT, size: 10 }}, bodyFont: {{ family: FONT, size: 12 }}, padding: 12, titleColor: DIM, bodyColor: INK }};

new Chart(document.getElementById('cumChart'), {{
  type: 'line',
  data: {{ labels: days.map(d => d.slice(5)), datasets: [{{
    data: cum, borderColor: '#d4a574', fill: true, tension: 0.3, borderWidth: 2,
    pointRadius: 4, pointBackgroundColor: '#d4a574', pointBorderColor: '#0a0e14', pointBorderWidth: 2,
    backgroundColor: ctx => {{ const c = ctx.chart.ctx.createLinearGradient(0,0,0,380); c.addColorStop(0,'rgba(212,165,116,0.25)'); c.addColorStop(1,'rgba(212,165,116,0)'); return c; }}
  }}] }},
  options: {{ responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }}, tooltip: {{ ...tooltipStyle, callbacks: {{ label: c => '  $' + c.parsed.y.toFixed(2) }} }} }},
    scales: {{ x: {{ grid: {{ color: GRID }}, ticks: {{ font: {{ family: FONT, size: 10 }} }} }},
               y: {{ grid: {{ color: GRID }}, ticks: {{ font: {{ family: FONT, size: 10 }}, callback: v => '$' + v }} }} }} }}
}});

new Chart(document.getElementById('dailyChart'), {{
  type: 'bar',
  data: {{ labels: days.map(d => d.slice(5)), datasets: [{{ data: days.map(d => dailyMap[d]), backgroundColor: '#6ea8c4' }}] }},
  options: {{ responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }}, tooltip: {{ ...tooltipStyle, callbacks: {{ label: c => '  $' + c.parsed.y.toFixed(2) }} }} }},
    scales: {{ x: {{ grid: {{ color: GRID }}, ticks: {{ font: {{ family: FONT, size: 10 }} }} }},
               y: {{ grid: {{ color: GRID }}, ticks: {{ font: {{ family: FONT, size: 10 }}, callback: v => '$' + v }} }} }} }}
}});

const catTotals = {{}};
TX.forEach(t => {{ catTotals[t[3]] = (catTotals[t[3]] || 0) + t[2]; }});
const catEntries = Object.entries(catTotals).sort((a,b) => b[1] - a[1]);
const totalAll = catEntries.reduce((s,c) => s+c[1], 0);

new Chart(document.getElementById('catDonut'), {{
  type: 'doughnut',
  data: {{ labels: catEntries.map(c => c[0]),
    datasets: [{{ data: catEntries.map(c => c[1]),
      backgroundColor: catEntries.map(c => CAT_COLORS[c[0]] || '#4a5468'),
      borderColor: '#11161f', borderWidth: 2 }}] }},
  options: {{ responsive: true, maintainAspectRatio: false, cutout: '62%',
    plugins: {{ legend: {{ position: 'right', labels: {{ font: {{ family: 'Inter', size: 11 }}, color: DIM, padding: 10, boxWidth: 10, boxHeight: 10 }} }},
      tooltip: {{ ...tooltipStyle, callbacks: {{ label: c => '  $' + c.parsed.toFixed(2) + ' · ' + ((c.parsed/totalAll)*100).toFixed(1) + '%' }} }} }} }}
}});

const maxCat = catEntries[0][1];
const catList = document.getElementById('catList');
catEntries.forEach(([name, amt]) => {{
  const pct = (amt/totalAll)*100; const w = (amt/maxCat)*100;
  const row = document.createElement('div'); row.className = 'cat-row';
  row.innerHTML = `<span class="cat-name">${{name}}</span><div><div class="cat-bar-track"><div class="cat-bar-fill" style="width:${{w}}%; background:${{CAT_COLORS[name] || '#4a5468'}};"></div></div><span class="cat-pct">${{pct.toFixed(1)}}% of total</span></div><span class="cat-amount">$${{amt.toFixed(2)}}</span>`;
  catList.appendChild(row);
}});

const merchTotals = {{}};
TX.forEach(t => {{
  const key = t[1].toLowerCase().startsWith('amzn') ? 'Amazon' : t[1];
  merchTotals[key] = (merchTotals[key] || 0) + t[2];
}});
const merchEntries = Object.entries(merchTotals).sort((a,b) => b[1] - a[1]).slice(0, 10);

new Chart(document.getElementById('merchChart'), {{
  type: 'bar',
  data: {{ labels: merchEntries.map(m => m[0]), datasets: [{{ data: merchEntries.map(m => m[1]), backgroundColor: '#d4a574' }}] }},
  options: {{ indexAxis: 'y', responsive: true, maintainAspectRatio: false,
    plugins: {{ legend: {{ display: false }}, tooltip: {{ ...tooltipStyle, callbacks: {{ label: c => '  $' + c.parsed.x.toFixed(2) }} }} }},
    scales: {{ x: {{ grid: {{ color: GRID }}, ticks: {{ font: {{ family: FONT, size: 10 }}, callback: v => '$' + v }} }},
               y: {{ grid: {{ display: false }}, ticks: {{ font: {{ family: 'Inter', size: 12 }}, color: INK }} }} }} }}
}});

const filtersEl = document.getElementById('filters');
const bodyEl = document.getElementById('txBody');
const cats = ['All', ...catEntries.map(c => c[0])];
cats.forEach(c => {{
  const b = document.createElement('button');
  b.className = 'chip' + (c === 'All' ? ' active' : '');
  b.textContent = c;
  b.onclick = () => {{
    document.querySelectorAll('.chip').forEach(x => x.classList.remove('active'));
    b.classList.add('active');
    document.querySelectorAll('#txBody tr').forEach(r => {{ r.classList.toggle('hidden', c !== 'All' && r.dataset.cat !== c); }});
  }};
  filtersEl.appendChild(b);
}});

const txSorted = [...TX].sort((a,b) => b[0].localeCompare(a[0]) || b[2] - a[2]);
txSorted.forEach(t => {{
  const tr = document.createElement('tr');
  tr.dataset.cat = t[3];
  const color = CAT_COLORS[t[3]] || '#4a5468';
  tr.innerHTML = `<td class="date">${{t[0].slice(5)}}</td><td>${{t[1]}}</td><td><span class="cat-tag" style="color:${{color}}; border-color:${{color}}33;">${{t[3]}}</span></td><td class="num">$${{t[2].toFixed(2)}}</td>`;
  bodyEl.appendChild(tr);
}});
</script>
</body>
</html>'''

with open(OUTPUT_HTML, 'w', encoding='utf-8') as f:
    f.write(html)

print(f'Wrote {OUTPUT_HTML} ({len(html)/1024:.1f} KB)')
print(f'Open it in any browser to view the interactive dashboard.')

## 10. Cleanup

Remove the temp frames directory.

In [ ]:
shutil.rmtree(frames_dir, ignore_errors=True)
print('Cleaned up temp frames.')

## Tuning tips

If the extraction looks off, try:

- **Missing transactions** → drop `VOTE_THRESHOLD` to 1, or increase `FRAMES_PER_SECOND` to 3 or 4. Scroll slowly when recording.
- **Duplicates with slightly different merchant names** → tighten the `ICON_JUNK` regex, or add a normalization step for the offending merchant.
- **Wrong dates** → check that the recording timestamp is correct (set `RECORDING_DATE_OVERRIDE`), and ensure the video starts with a visible date header.
- **Categories all 'Other'** → extend `CATEGORY_RULES` with your common merchants.
- **OCR garbage on a different bank's UI** → adjust `TX_LINE` and `DATE_HEADER` regexes to match that bank's format. Print the raw OCR text from a few frames to see what you're working with.